# Filtering quickstart — the fast exact NGH-MSM filter

This notebook shows how to **filter** a Gaussian switching system with the
package's fast, linear-time *exact* filter (NGH-MSM-KF, `GSSFilter` mode
`"h5_exact"`).

Given observations $Y_{1:n}$ the filter returns, recursively and in time linear
in $n$:

- $\mathbb{E}[X_n \mid Y_{1:n}]$ — the filtered state estimate,
- $\mathrm{Var}[X_n \mid Y_{1:n}]$ — its posterior covariance,
- $p(R_n \mid Y_{1:n})$ — the filtered regime probabilities.

We use the paper's scalar model **M1** ($K=2$ regimes, $q=s=1$). The simulation
study behind these models is in `docs/wojciech/rapport_experimental/`.

> Filtering only — this notebook does **not** cover parameter learning (EM).

## Setup

We import the public filtering API and define three one-line helpers: build a
paper model, simulate a trajectory, and run the filter step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from prg.classes.GSSSimulator import GSSSimulator
from prg.filter.gss_filter import GSSFilter
from prg.experiments.models_paper import get_params
from prg.experiments.run_simulations import _params_from_dict
from prg.experiments.reference_filters import exact_mixture_filter, with_stationary_init


def build_model(name):
    """Paper model "M1"/"M2"/"M3" as validated NGH-MSM params, started from its
    stationary law (so the exact filter is exact from the very first step)."""
    return with_stationary_init(_params_from_dict(get_params(name)))


def simulate(params, N, seed):
    """Sample a trajectory; return true regimes r, states X, observations Y."""
    rs, xs, ys = [], [], []
    for n, r, x, y in GSSSimulator(params, N=N, seed=seed):
        rs.append(int(r)); xs.append(np.ravel(x)); ys.append(np.ravel(y))
    return np.array(rs), np.array(xs), np.array(ys)


def run_filter(params, ys, mode="h5_exact"):
    """Run GSSFilter step by step; return E[X|y] (N,q), Var[X|y] (N,q,q), p(r|y) (N,K)."""
    filt = GSSFilter(params, mode=mode)
    Ex, Var, Pi = [], [], []
    for y in ys:
        res = filt.step(np.asarray(y, float).reshape(params.s, 1))
        Ex.append(res.E_x.ravel()); Var.append(np.asarray(res.Var_x)); Pi.append(np.ravel(res.pi))
    return np.array(Ex), np.array(Var), np.array(Pi)

## 1. Build a model

`build_model("M1")` returns validated NGH-MSM parameters: $K=2$ regimes, a scalar
hidden state ($q=1$) observed through a scalar ($s=1$). For your own models,
define a class under `prg/models/` and use `GSSParams.from_model(YourModel())`,
or run the CLI `python -m prg.filter.main --model <name> -N 1000`.

In [ ]:
params = build_model("M1")
print(f"K={params.K} regimes, q={params.q} (state dim), s={params.s} (obs dim)")

## 2. Simulate a trajectory

`GSSSimulator` yields `(n, r, x, y)` per step. We keep the ground-truth regimes
`rs` and states `xs` to score the filter; the filter itself only ever sees `ys`.

In [ ]:
rs, xs, ys = simulate(params, N=300, seed=7)
print("regimes:", rs.shape, " states:", xs.shape, " observations:", ys.shape)

## 3. Run the filter

The core call is `GSSFilter(params, mode="h5_exact").step(y)`, returning a
`FilterResult` with `.E_x` $(q,1)$, `.Var_x` $(q,q)$, `.pi` $(K,)$, plus
`.innovation` and `.log_lik`. The filter is causal: each estimate uses only
$Y_{1:n}$.

`mode="h5_exact"` is the fast **exact** filter; it requires the AB/(H5)
constraint, which the paper models satisfy. For an arbitrary GSS model that does
**not** satisfy (H5), use `mode="imm_general"` instead.

In [ ]:
Ex, Var, Pi = run_filter(params, ys)

# one FilterResult, to show the fields
filt = GSSFilter(params, mode="h5_exact")
res = filt.step(np.asarray(ys[0], float).reshape(params.s, 1))
print("step 0:  E[X|y] =", res.E_x.ravel(),
      "| p(r|y) =", np.round(res.pi, 3),
      "| log p(y_1) =", round(res.log_lik, 3))

## 4. Inspect the estimates

Top: the filtered mean tracks the hidden state, with a $\pm2\sigma$ band from the
posterior variance. Bottom: the filtered regime posterior $p(R_n{=}1\mid Y_{1:n})$
follows the true regime.

In [ ]:
sigma = np.sqrt(np.clip(Var[:, 0, 0], 0, None))
T = 150
n = np.arange(T)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
ax1.plot(n, xs[:T, 0], "k.", ms=4, label="true state $X_n$")
ax1.plot(n, Ex[:T, 0], color="#1f77b4", lw=1.3, label=r"$E[X_n\mid y_{1:n}]$")
ax1.fill_between(n, Ex[:T, 0] - 2 * sigma[:T], Ex[:T, 0] + 2 * sigma[:T],
                 color="#1f77b4", alpha=0.2, label=r"$\pm2\sigma$")
ax1.set_ylabel("state"); ax1.set_title("M1 — filtered state"); ax1.legend(fontsize=8, ncol=3)
ax2.plot(n, Pi[:T, 1], color="#1f77b4", lw=1.3, label=r"$p(R_n{=}1\mid y_{1:n})$")
ax2.plot(n, rs[:T], "k.", ms=4, label="true regime")
ax2.set_xlabel("$n$"); ax2.set_ylabel("regime"); ax2.legend(fontsize=8)
fig.tight_layout()

rmse = float(np.sqrt(np.mean((Ex[:, 0] - xs[:, 0]) ** 2)))
acc = float(np.mean(Pi.argmax(1) == rs))
print(f"state RMSE = {rmse:.3f}   |   regime accuracy = {acc:.0%}")

## 5. The filter is *exact*

NGH-MSM-KF is not an approximation: under the AB/(H5) constraint it returns the
true Bayesian posterior. We check this against the brute-force filter that
enumerates all $K^N$ regime histories (tractable only for small $N$).

In [ ]:
_, _, ys_short = simulate(params, N=8, seed=3)
Ex_ngh, _, _ = run_filter(params, ys_short)
Ex_exact, _, _ = exact_mixture_filter(params, ys_short)
print("max |NGH-MSM-KF - brute-force| on E[X|y]:",
      f"{np.max(np.abs(Ex_ngh - Ex_exact)):.2e}   (floating-point round-off)")

## Recap

```python
params = build_model("M1")                 # validated NGH-MSM params
rs, xs, ys = simulate(params, N=300, seed=7)
Ex, Var, Pi = run_filter(params, ys)       # E[X|y], Var[X|y], p(r|y)
```

- Core call: `GSSFilter(params, mode="h5_exact").step(y)` -> `FilterResult`.
- Batch alternative: `GSSFilter(params).run(N=1000, seed=42)` simulates **and**
  filters and writes a CSV (`E_x_i, V_x_i, p_r_k, sq_err`).
- For a model that does **not** satisfy (H5), use `mode="imm_general"`.
- Next: `02_filters_comparison.ipynb` — exactness across models, the value of
  modelling the switching, and multivariate tracking.